<a href="https://colab.research.google.com/github/TamPhu123/Assignment3_Lane-Detection/blob/main/EX05_(P0).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cập nhật kaggle lên version mới nhất (đang dùng 2.0.2, cần 2.2.2+)
!pip install --upgrade kaggle -q

# Restart lại environment để kaggle mới có hiệu lực
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_81b23fa70c37bc6db637807ad0eae17d'  # cần set lại sau restart

# Thử download
!kaggle competitions download -c paddy-disease-classification -p /content/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 5.7 MB/s eta 0:00:00
100% 1.02G/1.02G [00:09<00:00, 112MB/s]



In [2]:
import zipfile, os
from pathlib import Path

# Giải nén (tên file có thể khác tùy bạn tải cái nào)
zip_path = next(Path('/content/').glob('*.zip'))
print(f"Đang giải nén: {zip_path.name}")

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/paddy-raw/')

# Xem cấu trúc thư mục gốc
print("\n📁 Cấu trúc thư mục gốc:")
for p in sorted(Path('/content/paddy-raw/').rglob('*')):
    if p.is_dir():
        n_imgs = len(list(p.glob('*.[jJpP][pPnN][gGeE]*')))
        if n_imgs > 0:
            print(f"  {p.relative_to('/content/paddy-raw/')} ({n_imgs} ảnh)")

Đang giải nén: paddy-disease-classification.zip

📁 Cấu trúc thư mục gốc:
  test_images (3469 ảnh)
  train_images/bacterial_leaf_blight (479 ảnh)
  train_images/bacterial_leaf_streak (380 ảnh)
  train_images/bacterial_panicle_blight (337 ảnh)
  train_images/blast (1738 ảnh)
  train_images/brown_spot (965 ảnh)
  train_images/dead_heart (1442 ảnh)
  train_images/downy_mildew (620 ảnh)
  train_images/hispa (1594 ảnh)
  train_images/normal (1764 ảnh)
  train_images/tungro (1088 ảnh)


In [3]:
import shutil
from pathlib import Path

# Xóa toàn bộ thư mục cũ
dst_root = Path('/content/rice-disease/raw/')
if dst_root.exists():
    shutil.rmtree(dst_root)
    print("🗑️  Đã xóa thư mục cũ")

src_root = Path('/content/paddy-raw/train_images')
dst_root.mkdir(parents=True, exist_ok=True)

moved = 0
for src_folder in sorted(src_root.iterdir()):
    if not src_folder.is_dir():
        continue

    # Đổi 'normal' → 'healthy' ngay tại đây cho sạch
    folder_name = 'healthy' if src_folder.name == 'normal' else src_folder.name
    dst_folder = dst_root / folder_name
    dst_folder.mkdir(exist_ok=True)

    for img in src_folder.glob('*'):
        if img.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}:
            shutil.copy2(img, dst_folder / img.name)
            moved += 1

print(f"✅ Đã copy {moved:,} ảnh")
print("Folders:", sorted([p.name for p in dst_root.iterdir() if p.is_dir()]))

✅ Đã copy 10,407 ảnh
Folders: ['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'healthy', 'hispa', 'tungro']


In [4]:
from collections import Counter
from pathlib import Path

root = Path('/content/rice-disease/raw/')
counts = Counter(p.parent.name for p in root.rglob('*') if p.is_file())

print(f"{'Class':<30} {'Số ảnh':>8}")
print('─' * 42)
for cls, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"{cls:<30} {n:>8,}")
print('─' * 42)
print(f"{'TỔNG':<30} {sum(counts.values()):>8,}")

# Kiểm tra không còn 'normal' nữa
assert 'normal' not in counts, "❌ Vẫn còn folder 'normal', kiểm tra lại!"
assert 'other_unknown' not in counts, "❌ Vẫn còn 'other_unknown' từ lần chạy cũ!"
print("\n✅ Cấu trúc sạch, sẵn sàng sang Step 1")

Class                            Số ảnh
──────────────────────────────────────────
healthy                           1,764
blast                             1,738
hispa                             1,594
dead_heart                        1,442
tungro                            1,088
brown_spot                          965
downy_mildew                        620
bacterial_leaf_blight               479
bacterial_leaf_streak               380
bacterial_panicle_blight            337
──────────────────────────────────────────
TỔNG                             10,407

✅ Cấu trúc sạch, sẵn sàng sang Step 1


In [5]:
from collections import Counter
from pathlib import Path

root = Path('/content/rice-disease/raw/')
counts = Counter(p.parent.name for p in root.rglob('*') if p.is_file())

MISSING_FROM_SOURCE = ['sheath_blight', 'false_smut']

print(f"{'Class':<30} {'Số ảnh':>8}  {'Trạng thái'}")
print('─' * 60)
for cls, n in sorted(counts.items(), key=lambda x: -x[1]):
    if n < 300:
        status = '⚠️  dưới mục tiêu 300'
    else:
        status = '✅'
    print(f"{cls:<30} {n:>8,}  {status}")

print('─' * 60)
print(f"{'TỔNG':<30} {sum(counts.values()):>8,}")

print(f"\n❌ Bệnh trong taxonomy nhưng THIẾU trong dataset này:")
for m in MISSING_FROM_SOURCE:
    print(f"   - {m}")

Class                            Số ảnh  Trạng thái
────────────────────────────────────────────────────────────
healthy                           1,764  ✅
blast                             1,738  ✅
hispa                             1,594  ✅
dead_heart                        1,442  ✅
tungro                            1,088  ✅
brown_spot                          965  ✅
downy_mildew                        620  ✅
bacterial_leaf_blight               479  ✅
bacterial_leaf_streak               380  ✅
bacterial_panicle_blight            337  ✅
────────────────────────────────────────────────────────────
TỔNG                             10,407

❌ Bệnh trong taxonomy nhưng THIẾU trong dataset này:
   - sheath_blight
   - false_smut


In [6]:
!pip install numpy pandas matplotlib Pillow scikit-learn -q

In [7]:
from pathlib import Path
import json
import re
import math
import hashlib
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, UnidentifiedImageError
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from IPython.display import display

# ---------- USER CONFIGURATION ----------
DATASET_ROOT = Path('/content/rice-disease/raw')   # ← SỬA
METADATA_CSV = None                                 # ← GIỮ NGUYÊN
OUTPUT_DIR   = Path('./reports/central_vietnam_eda') # ← GIỮ NGUYÊN

TARGET_COL        = 'primary_disease'
IMAGE_ID_COL      = 'image_id'
IMAGE_EXTENSIONS  = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
GROUP_PRIORITY    = ['field_id', 'capture_session_id', 'plant_id', 'collector_batch']
REQUIRED_METADATA = ['province', 'season']
IMAGE_SCAN_LIMIT  = None
RANDOM_SEED       = 42
HOLDOUT_SIZE      = 0.20

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUTPUT_DIR.resolve())

Output directory: /content/reports/central_vietnam_eda


In [8]:
CANONICAL_LABELS = {
    'healthy', 'blast', 'bacterial_leaf_blight', 'sheath_blight',
    'brown_spot', 'false_smut', 'bacterial_leaf_streak',
    'bacterial_panicle_blight', 'downy_mildew', 'tungro',
    'other_unknown', 'needs_expert_review',
}

LABEL_ALIASES = {
    'dao_on': 'blast', 'dao on': 'blast', 'blast': 'blast',
    'bac_la': 'bacterial_leaf_blight', 'bac la': 'bacterial_leaf_blight',
    'bacterial_blight': 'bacterial_leaf_blight',
    'bacterial leaf blight': 'bacterial_leaf_blight',
    'kho_van': 'sheath_blight', 'kho van': 'sheath_blight',
    'sheath_blight': 'sheath_blight',
    'dom_nau': 'brown_spot', 'dom nau': 'brown_spot',
    'brown_spot': 'brown_spot', 'brown spot': 'brown_spot',
    'thoi_be': 'false_smut', 'thoi be': 'false_smut',
    'false_smut': 'false_smut',
    'bacterial_leaf_streak': 'bacterial_leaf_streak',
    'bacterial panicle blight': 'bacterial_panicle_blight',
    'downy_mildew': 'downy_mildew', 'tungro': 'tungro',
    'healthy': 'healthy', 'normal': 'healthy',
    'other': 'other_unknown', 'unknown': 'other_unknown',
    'needs_review': 'needs_expert_review',
}

COLUMN_ALIASES = {
    'disease_class': TARGET_COL, 'class': TARGET_COL,
    'label': TARGET_COL, 'target': TARGET_COL,
    'filename': IMAGE_ID_COL, 'file_name': IMAGE_ID_COL,
    'image': IMAGE_ID_COL, 'path': 'image_path',
    'severity': 'severity_score',
}

def clean_token(value):
    if pd.isna(value): return np.nan
    value = str(value).strip().lower()
    return re.sub(r'\s+', ' ', value)

def canonicalize_label(value):
    value = clean_token(value)
    if pd.isna(value): return np.nan
    return LABEL_ALIASES.get(value, value.replace(' ', '_'))

def normalize_metadata_columns(df):
    out = df.copy()
    out.columns = [str(c).strip().lower() for c in out.columns]
    out = out.rename(columns={k: v for k, v in COLUMN_ALIASES.items() if k in out.columns})
    if IMAGE_ID_COL in out.columns:
        out[IMAGE_ID_COL] = out[IMAGE_ID_COL].astype(str).map(lambda x: Path(x).name)
    return out

def scan_folder_dataset(root: Path) -> pd.DataFrame:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'DATASET_ROOT does not exist: {root}.')
    rows = []
    for path in sorted(root.rglob('*')):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            relative_path = path.relative_to(root).as_posix()
            rows.append({
                IMAGE_ID_COL: path.name,
                'relative_path': relative_path,
                'image_path': str(path.resolve()),
                'folder_label': path.parent.name,
                'source_dataset': 'local_folder_scan',
            })
    if not rows:
        raise ValueError(f'No supported image files found below {root.resolve()}')
    return pd.DataFrame(rows)

def build_manifest(root: Path, metadata_csv=None) -> pd.DataFrame:
    manifest = scan_folder_dataset(root)
    manifest['raw_label'] = manifest['folder_label']
    manifest[TARGET_COL] = manifest['folder_label'].map(canonicalize_label)
    if metadata_csv is not None:
        metadata_csv = Path(metadata_csv)
        meta = normalize_metadata_columns(pd.read_csv(metadata_csv))
        if meta[IMAGE_ID_COL].duplicated().any():
            dupes = meta.loc[meta[IMAGE_ID_COL].duplicated(keep=False), IMAGE_ID_COL].head(10).tolist()
            raise ValueError(f'Metadata image_id not unique: {dupes}')
        if TARGET_COL in meta.columns:
            meta['metadata_label'] = meta[TARGET_COL].map(canonicalize_label)
            meta = meta.drop(columns=[TARGET_COL])
        manifest = manifest.merge(meta, on=IMAGE_ID_COL, how='left', suffixes=('', '_meta'))
        if 'metadata_label' in manifest.columns:
            manifest[TARGET_COL] = manifest['metadata_label'].fillna(manifest[TARGET_COL])
            manifest['raw_label'] = manifest['metadata_label'].fillna(manifest['raw_label'])
            manifest = manifest.drop(columns=['metadata_label'])
    manifest['is_canonical_label'] = manifest[TARGET_COL].isin(CANONICAL_LABELS)
    manifest['label_needs_review'] = ~manifest['is_canonical_label'] | manifest[TARGET_COL].isin(['other_unknown', 'needs_expert_review'])
    manifest['image_uid'] = manifest['relative_path']
    return manifest

manifest = build_manifest(DATASET_ROOT, METADATA_CSV)
manifest.to_csv(OUTPUT_DIR / 'manifest_scanned.csv', index=False)
print(f'Images in manifest: {len(manifest):,}')
print('Manifest saved to:', OUTPUT_DIR / 'manifest_scanned.csv')
display(manifest.head())

Images in manifest: 10,407
Manifest saved to: reports/central_vietnam_eda/manifest_scanned.csv


,image_id,relative_path,image_path,folder_label,source_dataset,raw_label,primary_disease,is_canonical_label,label_needs_review,image_uid
0,100023.jpg,bacterial_leaf_blight/100023.jpg,/content/rice-disease/raw/bacterial_leaf_bligh...,bacterial_leaf_blight,local_folder_scan,bacterial_leaf_blight,bacterial_leaf_blight,True,False,bacterial_leaf_blight/100023.jpg
1,100049.jpg,bacterial_leaf_blight/100049.jpg,/content/rice-disease/raw/bacterial_leaf_bligh...,bacterial_leaf_blight,local_folder_scan,bacterial_leaf_blight,bacterial_leaf_blight,True,False,bacterial_leaf_blight/100049.jpg
2,100126.jpg,bacterial_leaf_blight/100126.jpg,/content/rice-disease/raw/bacterial_leaf_bligh...,bacterial_leaf_blight,local_folder_scan,bacterial_leaf_blight,bacterial_leaf_blight,True,False,bacterial_leaf_blight/100126.jpg
3,100133.jpg,bacterial_leaf_blight/100133.jpg,/content/rice-disease/raw/bacterial_leaf_bligh...,bacterial_leaf_blight,local_folder_scan,bacterial_leaf_blight,bacterial_leaf_blight,True,False,bacterial_leaf_blight/100133.jpg
4,100148.jpg,bacterial_leaf_blight/100148.jpg,/content/rice-disease/raw/bacterial_leaf_bligh...,bacterial_leaf_blight,local_folder_scan,bacterial_leaf_blight,bacterial_leaf_blight,True,False,bacterial_leaf_blight/100148.jpg


In [9]:
print('--- BASIC INTEGRITY ---')
print('Unique image rows :', manifest['image_uid'].nunique(), '/', len(manifest))
print('Duplicate filenames:', int(manifest[IMAGE_ID_COL].duplicated().sum()))
print('Duplicate rel paths:', int(manifest['relative_path'].duplicated().sum()))
print('Labels needing review:', int(manifest['label_needs_review'].sum()))

label_counts = (manifest[TARGET_COL]
                .value_counts(dropna=False)
                .rename_axis(TARGET_COL)
                .reset_index(name='n_images'))
label_counts['share'] = (label_counts['n_images'] / len(manifest)).round(3)
display(label_counts)

review_labels = sorted(manifest.loc[~manifest['is_canonical_label'], TARGET_COL].dropna().unique())
if review_labels:
    print('\n⚠️  Labels cần agronomist review:', review_labels)
else:
    print('\n✅ Không có label nào ngoài canonical list')

available_metadata = [
    c for c in [
        'province', 'season', 'month', 'growth_stage', 'field_id',
        'capture_session_id', 'severity_score', 'weather_condition'
    ] if c in manifest.columns
]

if available_metadata:
    miss = (manifest[available_metadata].isna().mean() * 100).round(1)
    display(miss.rename('missing_%').to_frame())
else:
    print('\n⚠️  Không có metadata context (province, season, field_id...)')
    print('   → Ghi vào báo cáo EX05 là limitation của source dataset này')

for col in REQUIRED_METADATA:
    if col not in manifest.columns or manifest[col].isna().all():
        print(f'⚠️  WARNING: `{col}` hoàn toàn thiếu — không thể làm regional holdout')

--- BASIC INTEGRITY ---
Unique image rows : 10407 / 10407
Duplicate filenames: 0
Duplicate rel paths: 0
Labels needing review: 3036


,primary_disease,n_images,share
0,healthy,1764,0.170
1,blast,1738,0.167
2,hispa,1594,0.153
3,dead_heart,1442,0.139
4,tungro,1088,0.105
5,brown_spot,965,0.093
6,downy_mildew,620,0.060
7,bacterial_leaf_blight,479,0.046
8,bacterial_leaf_streak,380,0.037
9,bacterial_panicle_blight,337,0.032



⚠️  Labels cần agronomist review: ['dead_heart', 'hispa']

⚠️  Không có metadata context (province, season, field_id...)
   → Ghi vào báo cáo EX05 là limitation của source dataset này
⚠️  WARNING: `province` hoàn toàn thiếu — không thể làm regional holdout
⚠️  WARNING: `season` hoàn toàn thiếu — không thể làm regional holdout


In [10]:
from PIL import Image, ImageOps, UnidentifiedImageError
import hashlib
import numpy as np
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm

def dhash(image, hash_size=8):
    gray = ImageOps.grayscale(image).resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
    pixels = np.asarray(gray, dtype=np.int16)
    bits = pixels[:, 1:] > pixels[:, :-1]
    bit_string = ''.join('1' if b else '0' for b in bits.flatten())
    return f'{int(bit_string, 2):0{hash_size * hash_size // 4}x}'

def image_record(path):
    path = Path(path)
    record = {'is_readable': False, 'error': None}
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            img = img.convert('RGB')
            arr = np.asarray(img)
            h, w = arr.shape[:2]
            gray = 0.299*arr[...,0] + 0.587*arr[...,1] + 0.114*arr[...,2]
            max_rgb = arr.max(axis=2).astype(float)
            min_rgb = arr.min(axis=2).astype(float)
            saturation = np.divide(max_rgb - min_rgb, np.maximum(max_rgb, 1.0))
            sharpness_proxy = float(np.var(np.diff(gray, axis=0)) + np.var(np.diff(gray, axis=1)))
            record.update({
                'is_readable': True,
                'width': int(w), 'height': int(h),
                'aspect_ratio': round(w/h, 4),
                'brightness_mean': round(float(gray.mean()), 3),
                'brightness_std': round(float(gray.std()), 3),
                'saturation_mean': round(float(saturation.mean()), 4),
                'sharpness_proxy': round(sharpness_proxy, 3),
                'file_bytes': path.stat().st_size,
                'sha1': hashlib.sha1(path.read_bytes()).hexdigest(),
                'dhash': dhash(img),
            })
    except Exception as exc:
        record['error'] = f'{type(exc).__name__}: {exc}'
    return record

# Quét toàn bộ ảnh — chạy lâu, có progress bar
records = []
paths = manifest['image_path'].tolist()

for p in tqdm(paths, desc='Quét ảnh'):
    records.append(image_record(p))

eda = pd.DataFrame(records, index=manifest.index)
image_stats = manifest.join(eda)
image_stats.to_csv(OUTPUT_DIR / 'image_stats.csv', index=False)

n_unreadable = int((~eda['is_readable']).sum())
print(f'\n✅ Quét xong {len(eda):,} ảnh')
print(f'   Đọc được : {int(eda["is_readable"].sum()):,}')
print(f'   Lỗi/hỏng : {n_unreadable}')
if n_unreadable:
    print(eda[~eda['is_readable']][['error']].head())

Quét ảnh:   0%|          | 0/10407 [00:00<?, ?it/s]


✅ Quét xong 10,407 ảnh
   Đọc được : 10,407
   Lỗi/hỏng : 0


In [11]:
readable = image_stats[image_stats['is_readable']].copy()

exact_dups = readable[readable.duplicated('sha1', keep=False)].sort_values('sha1')
perceptual_dups = readable[readable.duplicated('dhash', keep=False)].sort_values('dhash')

exact_dups.to_csv(OUTPUT_DIR / 'duplicate_exact_sha1.csv', index=False)
perceptual_dups.to_csv(OUTPUT_DIR / 'duplicate_perceptual_dhash_candidates.csv', index=False)

print(f'Exact duplicate rows     : {len(exact_dups)}')
print(f'Perceptual dup candidates: {len(perceptual_dups)}')

if len(exact_dups):
    display(exact_dups[[IMAGE_ID_COL, 'relative_path', TARGET_COL, 'sha1']].head(10))

# Collection shortcut check: so sánh brightness/sharpness theo class
shortcut_cols = [c for c in ['brightness_mean', 'saturation_mean', 'sharpness_proxy', 'width', 'height'] if c in readable.columns]
shortcut_summary = readable.groupby(TARGET_COL)[shortcut_cols].agg(['mean', 'std']).round(2)
print('\n--- Image quality by class (shortcut check) ---')
display(shortcut_summary)
shortcut_summary.to_csv(OUTPUT_DIR / 'collection_shortcut_summary.csv')

Exact duplicate rows     : 146
Perceptual dup candidates: 255


,image_id,relative_path,primary_disease,sha1
6215,101562.jpg,healthy/101562.jpg,healthy,08850a875edcd54c75533908de9e743e855aaead
7519,109156.jpg,healthy/109156.jpg,healthy,08850a875edcd54c75533908de9e743e855aaead
4142,101868.jpg,dead_heart/101868.jpg,dead_heart,0973cc6e79890b5fe161a8f5874022b224cc02ed
5279,110000.jpg,dead_heart/110000.jpg,dead_heart,0973cc6e79890b5fe161a8f5874022b224cc02ed
4761,106481.jpg,dead_heart/106481.jpg,dead_heart,0973cc6e79890b5fe161a8f5874022b224cc02ed
2732,109346.jpg,blast/109346.jpg,blast,0cbe1d803e190139e0d62243162fccb25eee7617
1907,104021.jpg,blast/104021.jpg,blast,0cbe1d803e190139e0d62243162fccb25eee7617
9905,105747.jpg,tungro/105747.jpg,tungro,1371b76c1ffb12f0e212b29d5f8cacf71463a846
9629,103091.jpg,tungro/103091.jpg,tungro,1371b76c1ffb12f0e212b29d5f8cacf71463a846
7036,106366.jpg,healthy/106366.jpg,healthy,1b7bd7f617bbc1a56d5381ede3c2936aeb7a8a48



--- Image quality by class (shortcut check) ---


brightness_mean        saturation_mean        \
                                    mean    std            mean   std   
primary_disease                                                         
bacterial_leaf_blight             135.40  11.19            0.66  0.15   
bacterial_leaf_streak             122.58   8.78            0.61  0.14   
bacterial_panicle_blight          125.34  12.96            0.50  0.22   
blast                             129.08  14.40            0.66  0.19   
brown_spot                        134.32  12.89            0.61  0.16   
dead_heart                        127.22  13.65            0.60  0.15   
downy_mildew                      132.69  13.35            0.65  0.12   
healthy                           137.24  11.55            0.67  0.11   
hispa                             136.02   9.50            0.65  0.15   
tungro                            136.18  12.75            0.67  0.13   

                         sharpness_proxy           width         height         
                                    mean     std    mean    std    mean    std  
primary_disease                                                                 
bacterial_leaf_blight            1124.83  441.89  480.67  10.33  639.33  10.33  
bacterial_leaf_streak            1204.01  539.35  480.00   0.00  640.00   0.00  
bacterial_panicle_blight          650.76  435.08  480.00   0.00  640.00   0.00  
blast                             896.18  515.07  480.00   0.00  640.00   0.00  
brown_spot                       1031.89  438.76  480.33   7.28  639.67   7.28  
dead_heart                        870.06  418.56  480.00   0.00  640.00   0.00  
downy_mildew                     1020.83  456.09  480.00   0.00  640.00   0.00  
healthy                          1051.28  520.60  480.00   0.00  640.00   0.00  
hispa                             934.88  541.15  480.00   0.00  640.00   0.00  
tungro                           1100.00  499.33  480.00   0.00  640.00   0.00

In [12]:
# Xóa exact duplicates — giữ bản đầu tiên mỗi nhóm
manifest_deduped = image_stats[image_stats['is_readable']].copy()
before = len(manifest_deduped)

manifest_deduped = manifest_deduped.drop_duplicates(subset='sha1', keep='first')
after = len(manifest_deduped)

print(f'Trước khi dedup : {before:,}')
print(f'Sau khi dedup   : {after:,}')
print(f'Đã xóa          : {before - after} exact duplicates')
print(f'Còn lại         : {after:,} ảnh')

Trước khi dedup : 10,407
Sau khi dedup   : 10,333
Đã xóa          : 74 exact duplicates
Còn lại         : 10,333 ảnh


In [13]:
import json
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split

def proposed_holdout_split(frame, target_col=TARGET_COL, test_size=HOLDOUT_SIZE,
                            seed=RANDOM_SEED, trials=50):
    data = frame.dropna(subset=[target_col]).copy()
    global_dist = data[target_col].value_counts(normalize=True)

    # Không có field_id — dùng dhash làm proxy group
    # Ảnh có cùng dhash (trông giống nhau) sẽ nằm cùng 1 split
    group_col = None
    if 'dhash' in data.columns and data['dhash'].nunique() < len(data):
        data['_group'] = data['dhash']
        group_col = '_group'
        print(f'Dùng dhash làm group proxy: {data[group_col].nunique()} groups từ {len(data)} ảnh')

    if group_col is not None:
        best = None
        for trial in range(trials):
            splitter = GroupShuffleSplit(n_splits=1, test_size=test_size,
                                         random_state=seed + trial)
            train_idx, test_idx = next(
                splitter.split(data, y=data[target_col], groups=data[group_col])
            )
            test = data.iloc[test_idx]
            test_dist = (test[target_col].value_counts(normalize=True)
                         .reindex(global_dist.index, fill_value=0))
            missing_classes = int((test_dist == 0).sum())
            balance_error = float((test_dist - global_dist).abs().sum())
            score = 100 * missing_classes + balance_error
            if best is None or score < best['score']:
                best = {'score': score, 'train_idx': train_idx, 'test_idx': test_idx,
                        'missing_classes': missing_classes, 'balance_error': balance_error}

        data['split'] = 'train'
        data.iloc[best['test_idx'], data.columns.get_loc('split')] = 'test'
        diagnostics = {
            'method': 'GroupShuffleSplit (dhash proxy — no field_id available)',
            'group_col': 'dhash',
            'missing_test_classes': best['missing_classes'],
            'class_distribution_error_l1': round(best['balance_error'], 4),
            'n_train': int((data['split'] == 'train').sum()),
            'n_test': int((data['split'] == 'test').sum()),
        }
    else:
        counts = data[target_col].value_counts()
        train_idx, test_idx = train_test_split(
            np.arange(len(data)), test_size=test_size,
            random_state=seed, stratify=data[target_col]
        )
        data['split'] = 'train'
        data.iloc[test_idx, data.columns.get_loc('split')] = 'test'
        diagnostics = {
            'method': 'Stratified random split (WARNING: no group key)',
            'group_col': None,
            'missing_test_classes': 0,
            'class_distribution_error_l1': None,
            'n_train': int((data['split'] == 'train').sum()),
            'n_test': int((data['split'] == 'test').sum()),
        }
        print('⚠️  WARNING: Không có group key — split này chỉ dùng cho EDA, không dùng cho training thật')

    return data, diagnostics

split_manifest, split_diagnostics = proposed_holdout_split(manifest_deduped)
split_manifest.to_csv(OUTPUT_DIR / 'proposed_group_holdout.csv', index=False)

print('\n--- Split diagnostics ---')
print(json.dumps(split_diagnostics, indent=2))
print('\n--- Class distribution trong mỗi split ---')
display(pd.crosstab(split_manifest[TARGET_COL], split_manifest['split'], margins=True))

# Kiểm tra leakage: không có dhash nào xuất hiện ở cả train lẫn test
if 'dhash' in split_manifest.columns:
    leakage = split_manifest.groupby('dhash')['split'].nunique()
    n_leaked = int((leakage > 1).sum())
    print(f'\nGroups (dhash) xuất hiện ở cả 2 split: {n_leaked}')
    if n_leaked == 0:
        print('✅ Không có leakage')
    else:
        print('⚠️  Có leakage — chạy lại với trials cao hơn')

Dùng dhash làm group proxy: 10276 groups từ 10333 ảnh

--- Split diagnostics ---
{
  "method": "GroupShuffleSplit (dhash proxy \u2014 no field_id available)",
  "group_col": "dhash",
  "missing_test_classes": 0,
  "class_distribution_error_l1": 0.0255,
  "n_train": 8262,
  "n_test": 2071
}

--- Class distribution trong mỗi split ---


split,test,train,All
primary_disease,,,
bacterial_leaf_blight,92,379,471
bacterial_leaf_streak,73,307,380
bacterial_panicle_blight,70,266,336
blast,360,1368,1728
brown_spot,192,761,953
dead_heart,288,1141,1429
downy_mildew,123,495,618
healthy,346,1403,1749
hispa,303,1286,1589



Groups (dhash) xuất hiện ở cả 2 split: 0
✅ Không có leakage


In [14]:
summary = {
    'n_original_images': len(image_stats),
    'n_after_dedup': len(manifest_deduped),
    'n_exact_duplicates_removed': len(image_stats) - len(manifest_deduped),
    'n_perceptual_dup_candidates': len(perceptual_dups),
    'n_unreadable': 0,
    'n_classes_total': int(image_stats[TARGET_COL].nunique()),
    'n_classes_canonical': int(image_stats[image_stats['is_canonical_label']][TARGET_COL].nunique()),
    'labels_needing_review': ['dead_heart', 'hispa'],
    'missing_taxonomy': ['sheath_blight', 'false_smut'],
    'has_province_metadata': False,
    'has_season_metadata': False,
    'has_field_id': False,
    'split_method': split_diagnostics['method'],
    'n_train': split_diagnostics['n_train'],
    'n_test': split_diagnostics['n_test'],
    'missing_test_classes': split_diagnostics['missing_test_classes'],
    'limitations': [
        'Dataset từ Ấn Độ (Tamil Nadu), không phải Central Vietnam',
        'Không có metadata province/season/field_id',
        'sheath_blight và false_smut không có trong source dataset',
        'hispa và dead_heart chưa được agronomist xác nhận taxonomy',
        'Group split dùng dhash proxy thay vì field_id thật',
    ]
}

(OUTPUT_DIR / 'eda_summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print('\n--- GATE CHECKLIST ---')
print('Taxonomy  :', '⚠️  Chờ agronomist confirm hispa, dead_heart')
print('Integrity :', '✅ Không file hỏng, exact dup đã xóa')
print('Coverage  :', '⚠️  Thiếu sheath_blight, false_smut — documented')
print('Metadata  :', '❌ Không có province/season → không thể regional holdout')
print('Split     :', '✅ dhash proxy — không có leakage' if split_diagnostics['missing_test_classes'] == 0 else '⚠️  Có missing classes trong test')
print('Safety    :', '⚠️  Chưa định nghĩa low-confidence behavior')

{
  "n_original_images": 10407,
  "n_after_dedup": 10333,
  "n_exact_duplicates_removed": 74,
  "n_perceptual_dup_candidates": 255,
  "n_unreadable": 0,
  "n_classes_total": 10,
  "n_classes_canonical": 8,
  "labels_needing_review": [
    "dead_heart",
    "hispa"
  ],
  "missing_taxonomy": [
    "sheath_blight",
    "false_smut"
  ],
  "has_province_metadata": false,
  "has_season_metadata": false,
  "has_field_id": false,
  "split_method": "GroupShuffleSplit (dhash proxy — no field_id available)",
  "n_train": 8262,
  "n_test": 2071,
  "missing_test_classes": 0,
  "limitations": [
    "Dataset từ Ấn Độ (Tamil Nadu), không phải Central Vietnam",
    "Không có metadata province/season/field_id",
    "sheath_blight và false_smut không có trong source dataset",
    "hispa và dead_heart chưa được agronomist xác nhận taxonomy",
    "Group split dùng dhash proxy thay vì field_id thật"
  ]
}

--- GATE CHECKLIST ---
Taxonomy  : ⚠️  Chờ agronomist confirm hispa, dead_heart
Integrity : ✅ Không 

In [15]:
CAPTURE_GROUP_RULES = """
CAPTURE GROUP RULES — EX05 P0
==============================
Dataset source : Paddy Doctor (Kaggle competition paddy-disease-classification)
Origin         : Tamil Nadu, India — NOT Central Vietnam field data

Group definition:
  - Lý tưởng: field_id hoặc capture_session_id (không có trong dataset này)
  - Proxy dùng cho EX05: dhash (perceptual hash) — ảnh trông giống nhau
    được coi là cùng capture group và phải nằm trong cùng 1 split

Split rule:
  - Một dhash group KHÔNG được xuất hiện ở cả train lẫn test
  - Đã kiểm tra: 0 groups bị leakage ✅

Limitations cần ghi vào project log:
  1. dhash proxy kém chính xác hơn field_id thật
  2. Không có province/season → không thể làm regional holdout
  3. Khi có Central Vietnam field data thật, phải làm lại split với field_id

Taxonomy pending:
  - hispa: côn trùng (pest), không phải bệnh — chờ agronomist quyết định
  - dead_heart: triệu chứng do sâu đục thân — chờ agronomist quyết định
  - Hiện tại: giữ nguyên 2 class này trong manifest, KHÔNG đưa vào baseline training
"""

rules_path = OUTPUT_DIR / 'capture_group_rules.txt'
rules_path.write_text(CAPTURE_GROUP_RULES, encoding='utf-8')
print(CAPTURE_GROUP_RULES)
print(f"✅ Đã lưu: {rules_path}")


CAPTURE GROUP RULES — EX05 P0
Dataset source : Paddy Doctor (Kaggle competition paddy-disease-classification)
Origin         : Tamil Nadu, India — NOT Central Vietnam field data

Group definition:
  - Lý tưởng: field_id hoặc capture_session_id (không có trong dataset này)
  - Proxy dùng cho EX05: dhash (perceptual hash) — ảnh trông giống nhau
    được coi là cùng capture group và phải nằm trong cùng 1 split

Split rule:
  - Một dhash group KHÔNG được xuất hiện ở cả train lẫn test
  - Đã kiểm tra: 0 groups bị leakage ✅

Limitations cần ghi vào project log:
  1. dhash proxy kém chính xác hơn field_id thật
  2. Không có province/season → không thể làm regional holdout
  3. Khi có Central Vietnam field data thật, phải làm lại split với field_id

Taxonomy pending:
  - hispa: côn trùng (pest), không phải bệnh — chờ agronomist quyết định
  - dead_heart: triệu chứng do sâu đục thân — chờ agronomist quyết định
  - Hiện tại: giữ nguyên 2 class này trong manifest, KHÔNG đưa vào baseline training

In [18]:
from google.colab import drive
import shutil
import os

# 1. "Cắm USB" (Kết nối Google Drive)
# Hệ thống sẽ hiện popup yêu cầu bạn chọn tài khoản Gmail và bấm Cho phép (Allow)
drive.mount('/content/drive')

# 2. Định nghĩa đường dẫn
# Thư mục tạm đang chứa kết quả EX05 của bạn
source_dir = '/content/reports/central_vietnam_eda'

# Thư mục trên Google Drive của bạn (Tự động tạo thư mục Rice_Disease_Project)
dest_dir = '/content/drive/MyDrive/Rice_Disease_Project/reports/central_vietnam_eda'

# 3. Copy toàn bộ thành quả sang Drive
print("Đang copy dữ liệu sang Google Drive...")
shutil.copytree(source_dir, dest_dir, dirs_exist_ok=True)
print(f"✅ Đã lưu an toàn toàn bộ báo cáo vào Drive của bạn tại: {dest_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đang copy dữ liệu sang Google Drive...
✅ Đã lưu an toàn toàn bộ báo cáo vào Drive của bạn tại: /content/drive/MyDrive/Rice_Disease_Project/reports/central_vietnam_eda
